# Prepare SQuAD_tiny Dataset for Assignment 2

This code prepare SQuAD_tiny from the SQuAD dataset.

# 0. Import libraries

In [ ]:
%conda install -y -c conda-forge tiktoken sentencepiece datasets transformers rouge-score


In [2]:
import importlib.util
import torch
import numpy as np
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from rouge_score import rouge_scorer
from transformers import logging as transformers_logging

# T5's tokenizer and the pretrained comparison model both require sentencepiece,
# and tiktoken is used as transformers' fallback fast-tokenizer backend.
# transformers caches backend availability at import time, so installing
# these after this cell has already run in the current kernel will NOT
# be picked up without restarting the kernel - fail fast here with a clear
# message instead of a confusing "Could not extract SentencePiece model" error later on.
missing_packages = [pkg for pkg in ("sentencepiece", "tiktoken") if importlib.util.find_spec(pkg) is None]
if missing_packages:
    raise ImportError(
        f"Missing required package(s): {', '.join(missing_packages)}. "
        f"Run `conda install {' '.join(missing_packages)}`, then RESTART THE KERNEL "
        "(not just re-run this cell) before re-running the notebook, since "
        "transformers caches backend availability at import time."
    )


In [3]:
# Set seed for reproducibility
torch.manual_seed(42)

# 1. Load and preprocess SQuAD dataset

In [4]:
# 1. Load and preprocess SQuAD dataset
#dataset = load_dataset("squad")

dataset = load_dataset("rajpurkar/squad") # avoid error

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
# Take subsets to avoid overload
train_dataset = dataset["train"].select(range(10000,11000))
val_dataset = dataset["validation"].select(range(3000,3100))
test_dataset = dataset["validation"].select(range(3100, 3200))  # No official SQuAD test set

In [ ]:
print("Size of training set:", len(train_dataset))
print("Size of validation set:", len(val_dataset))
print("Size of testing set:", len(test_dataset))

Size of training set: 1000
Size of validation set: 100
Size of testing set: 100


In [ ]:
MODEL_NAME = "t5-small"
#MODEL_NAME = "t5-base"
MAX_INPUT_LENGTH = 512
MAX_OUTPUT_LENGTH = 128
# Load tokenizer and model
# Force the SentencePiece tokenizer; otherwise Transformers may try to parse
# spiece.model with the TikToken converter.
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
def encode_question_and_context(question, context):
    return f"question: {question}  context: {context}"

# Obtains the context, question and answer from a given sample.
def extract_sample_parts(sample):
    context = sample["context"]
    question = sample["question"]
    answer = sample["answers"]['text'][0]
    question_with_context = encode_question_and_context(question, context)
    return (question_with_context, question, answer)

# Encodes the sample, returning token IDs.
def preprocess(sample):
    # Extract data from sample.
    question_with_context, question, answer = extract_sample_parts(sample)

    # Using truncation causes the tokenizer to emit a warning for every sample.
    # This will generate a significant amount of messages, and likely crash
    # your browser tab. We temporarily disable log messages to work around this.
    # See https://github.com/huggingface/transformers/issues/14285
    old_level = transformers_logging.get_verbosity()
    transformers_logging.set_verbosity_error()

    # Generate tokens for the input. question_with_context already contains
    # both the question and the context, so it is tokenized on its own —
    # passing `question` again as a second (pair) argument would duplicate
    # it in the encoded input, since T5's tokenizer simply concatenates
    # pair sequences rather than using a BERT-style [SEP]/segment scheme.
    input_tokens = tokenizer(question_with_context, padding="max_length",
                             truncation=True, max_length=MAX_INPUT_LENGTH)

    # Generate tokens for the expected answer. There is no need to include the
    output_tokens = tokenizer(answer, padding="max_length", truncation=True,
                              max_length=MAX_OUTPUT_LENGTH)

    # Restore old logging level, see above.
    transformers_logging.set_verbosity(old_level)

    # The output of the tokenizer is a map containing {input_ids, attention_mask}.
    # For trianing, we need to add the labels (answer/output tokens) to the map.
    input_tokens["labels"] = np.array(output_tokens["input_ids"])

    return input_tokens

In [ ]:
# Preprocess the datasets
training_set_enc = train_dataset.map(preprocess, batched=False)
validation_set_enc = val_dataset.map(preprocess, batched=False)
testing_set_enc = test_dataset.map(preprocess, batched=False)

In [ ]:
# Prepare 20 data points for qualitative analysis
q_data = test_dataset.select(range(20))
q_data

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 20
})

In [ ]:
# Set the format of the datasets to PyTorch tensors
columns = ["input_ids", "attention_mask", "labels"]
training_set_enc.set_format(type="torch", columns=columns)
validation_set_enc.set_format(type="torch", columns=columns)
testing_set_enc.set_format(type="torch", columns=columns)

# hyperparameters for training
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=10
)

# Trainer initialization and start of training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=training_set_enc,
    eval_dataset=validation_set_enc,
    processing_class=tokenizer,  #  use processing_class instead of tokenizer
    data_collator=DataCollatorForSeq2Seq(tokenizer)
)

print("Starting Fine-tuning...")
trainer.train()

Starting Fine-tuning...


c:\Users\Dmitry.Novik\AppData\Local\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

Cell 2: Define generation and ROUGE calculation functions for model evaluation

In [ ]:
from itertools import batched
from rouge_score import rouge_scorer

# Generate a single response
def generate_response(tok, mod, question):
    # Use MAX_INPUT_LENGTH (not MAX_OUTPUT_LENGTH) here: this truncates the
    # *input* (question + context) being tokenized for generation, and must
    # match the length used during training preprocessing so long contexts
    # aren't silently cut short at inference time.
    tokenized = tok(question, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_LENGTH).to(mod.device)
    with torch.no_grad():
        outputs = mod.generate(**tokenized, max_new_tokens=MAX_OUTPUT_LENGTH)
    return tok.batch_decode(outputs, skip_special_tokens=True)

# Generate answers for the entire dataset (with or without context)
def generate_answers(tok, mod, dataset, use_context=True, limit=None):
    if limit is not None:
        dataset = dataset.select(range(limit))

    questions, inputs, references = [], [], []
    for sample in dataset:
        question_with_context, question, answer = extract_sample_parts(sample)
        inputs.append(question_with_context if use_context else question)
        questions.append(question)
        references.append(answer)

    outputs = []
    # Process in batches of 32 to manage GPU memory[cite: 3]
    for samples in batched(inputs, 32):
        responses = generate_response(tok, mod, list(samples))
        outputs.extend(responses)
    return outputs, references, questions

# Compute average ROUGE score[cite: 3]
def compute_average_score(scores, metric, key):
    total = sum(getattr(score[metric], key) for score in scores)
    return total / len(scores)

# Compute ROUGE score[cite: 3]
def compute_rouge(predictions, references):
    metrics = ["rouge1", "rouge2", "rougeL"]
    scorer = rouge_scorer.RougeScorer(metrics, use_stemmer=True)
    scores = [scorer.score(ref, pred) for pred, ref in zip(predictions, references)]

    results = {}
    for metric in metrics:
        for k in ["precision", "recall", "fmeasure"]:
            results[f"{metric}_{k}"] = round(compute_average_score(scores, metric, k), 4)
    return results

Cell 3: ROUGE performance evaluation and analysis of test set generation results

In [ ]:
# Generate test set answers with the fine-tuned model (with/without context comparison)[cite: 3]
answers_ctx, refs_ctx, questions_ctx = generate_answers(tokenizer, model, test_dataset, True)
answers_noctx, refs_noctx, questions_noctx = generate_answers(tokenizer, model, test_dataset, False)

# Print ROUGE scores[cite: 3]
print("ROUGE with context:", compute_rouge(answers_ctx, refs_ctx))
print("ROUGE without context:", compute_rouge(answers_noctx, refs_noctx))

# Analyze the first 5 generated samples[cite: 3]
print("\n*** Generative Analysis (First 5 samples) ***")
for i in range(5):
    print(f"[{i+1}] Question: {questions_ctx[i]}")
    print(f"Reference Answer: {refs_ctx[i]}")
    print(f"Prediction (WITH context): {answers_ctx[i]}")
    print(f"Prediction (WITHOUT context): {answers_noctx[i]}")
    print("-" * 60)

ROUGE with context: {'rouge1_precision': 0.7846, 'rouge1_recall': 0.7886, 'rouge1_fmeasure': 0.7623, 'rouge2_precision': 0.5357, 'rouge2_recall': 0.5322, 'rouge2_fmeasure': 0.5124, 'rougeL_precision': 0.7839, 'rougeL_recall': 0.7861, 'rougeL_fmeasure': 0.7612}
ROUGE without context: {'rouge1_precision': 0.0106, 'rouge1_recall': 0.0341, 'rouge1_fmeasure': 0.0156, 'rouge2_precision': 0.0, 'rouge2_recall': 0.0, 'rouge2_fmeasure': 0.0, 'rougeL_precision': 0.0106, 'rougeL_recall': 0.0341, 'rougeL_fmeasure': 0.0156}

*** Generative Analysis (First 5 samples) ***
[1] Question: What country initially received the largest number of Huguenot refugees?
Reference Answer: the Dutch Republic
Prediction (WITH context): Dutch Republic
Prediction (WITHOUT context): Quel pays a prim prim primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi primi pri

Cell 4: Comparison with a pre-trained model

In [ ]:
from transformers import AutoModelForSeq2SeqLM

pt_model_name = "mrm8488/t5-base-finetuned-squadv2"

# Use T5Tokenizer directly instead of AutoTokenizer. AutoTokenizer's
# use_fast=False is not reliably honored in this transformers version: it
# still routes through the newer TokenizersBackend conversion path, which -
# if it hits any error loading spiece.model via sentencepiece - silently
# falls back to parsing it as a TikToken BPE file and crashes with an
# unrelated "not enough values to unpack" ValueError. T5Tokenizer loads the
# SentencePiece model directly and either works or raises a clear error.
pt_tokenizer = T5Tokenizer.from_pretrained(pt_model_name, use_fast=False)
pt_model = AutoModelForSeq2SeqLM.from_pretrained(pt_model_name).to(model.device)

# Evaluate the test set with the pre-trained model
pt_answers_ctx, pt_refs_ctx, _ = generate_answers(pt_tokenizer, pt_model, test_dataset, True)
print("Pre-trained Model ROUGE (with context):", compute_rouge(pt_answers_ctx, pt_refs_ctx))

In [ ]:
# Pre-trained model - evaluate without context (replicating the Task 3/4 evaluation approach)
pt_answers_noctx, pt_refs_noctx, pt_questions_noctx = generate_answers(pt_tokenizer, pt_model, test_dataset, False)
print("Pre-trained Model ROUGE (without context):", compute_rouge(pt_answers_noctx, pt_refs_noctx))

# Pre-trained model - generative analysis of the first 5 samples (with/without context comparison)
print("\n*** Pre-trained Model Generative Analysis (First 5 samples) ***")
for i in range(5):
    print(f"[{i+1}] Question: {pt_questions_noctx[i]}")
    print(f"Reference Answer: {pt_refs_ctx[i]}")
    print(f"Prediction (WITH context): {pt_answers_ctx[i]}")
    print(f"Prediction (WITHOUT context): {pt_answers_noctx[i]}")
    print("-" * 60)